# 🔍 Image Forgery Detection — IMD2020 Dataset
### ResNet34 + ELA Model — Resuming from Previous Checkpoint

This notebook:
1. Installs all dependencies
2. Downloads the IMD2020 dataset (Real + Inpainting + Masks)
3. Builds the **ForgeryClassifier** (ResNet34 encoder via SMP)
4. **Loads your previous checkpoint** (`best_forgery_model (1).pth`) to resume training
5. Continues training for **5 more epochs** on IMD2020
6. Evaluates on test set with full metrics
7. Saves best model + confusion matrix

> **Step 0 — Before running:** Upload `best_forgery_model (1).pth` to Colab using the 📁 Files panel on the left sidebar.  
> **Runtime**: Set to **GPU** (Runtime → Change Runtime Type → T4 GPU)

## 1. Install Dependencies

In [ ]:
%%capture
!pip install opencv-python-headless pillow scikit-learn
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install segmentation-models-pytorch albumentations
!pip install seaborn matplotlib tqdm
print('All dependencies installed.')

## 2. Import Libraries

In [ ]:
import os
import io
import cv2
import zipfile
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageChops, ImageEnhance
from tqdm.notebook import tqdm
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import segmentation_models_pytorch as smp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Using device: cpu


## 3. Download IMD2020 Dataset

We download:
- **4 zips** of real images (~35,000 images from 2,322 cameras)
- **7 zips** of inpainting (tampered) images (~35,000)
- **1 zip** of binary masks for inpainting images

Total download: ~15–20 GB. This will take ~10–20 minutes on Colab.

In [ ]:
BASE_URL  = 'https://staff.utia.cas.cz/novozada/db/'
DATA_ROOT = '/content/IMD2020'
REAL_DIR  = f'{DATA_ROOT}/real'
FAKE_DIR  = f'{DATA_ROOT}/tampered'
MASK_DIR  = f'{DATA_ROOT}/masks'

os.makedirs(REAL_DIR, exist_ok=True)
os.makedirs(FAKE_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

import ssl # Import ssl module
from tqdm.notebook import tqdm # Import tqdm

def download_and_extract(url, extract_to, desc=''):
    """
    Downloads a zip file from a given URL, extracts its contents,
    and saves it to a specified directory.
    """
    fname = url.split('/')[-1]
    local = f'/tmp/{fname}'
    if not os.path.exists(local):
        print(f'  Downloading {fname} ...')
        # Use ssl._create_unverified_context() to bypass SSL verification
        with urllib.request.urlopen(url, context=ssl._create_unverified_context()) as response, open(local, 'wb') as out_file:
            # Get total file size from headers if available
            total_size = int(response.headers.get('content-length', 0))
            block_size = 8192 # 8KB chunks
            with tqdm(total=total_size, unit='B', unit_scale=True, desc=f'    {fname}') as pbar:
                for chunk in iter(lambda: response.read(block_size), b''):
                    out_file.write(chunk)
                    pbar.update(len(chunk))
    else:
        print(f'  {fname} already cached.')
    print(f'  Extracting {fname} \u2192 {extract_to} ...')
    with zipfile.ZipFile(local, 'r') as z:
        z.extractall(extract_to)
    print(f'  \u2705 Done: {fname}')

# \u2500\u2500 Real images (4 parts) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
REAL_ZIPS = [
    BASE_URL + 'IMD2020_real_01.zip',
    BASE_URL + 'IMD2020_real_02.zip',
    BASE_URL + 'IMD2020_real_03.zip',
    BASE_URL + 'IMD2020_real_04.zip',
]

# \u2500\u2500 Inpainting / tampered images (7 parts) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
FAKE_ZIPS = [
    BASE_URL + 'IMD2020_Generative_Image_Inpainting_yu2018_01.zip',
    BASE_URL + 'IMD2020_Generative_Image_Inpainting_yu2018_02.zip',
    BASE_URL + 'IMD2020_Generative_Image_Inpainting_yu2018_03.zip',
    BASE_URL + 'IMD2020_Generative_Image_Inpainting_yu2018_04.zip',
    BASE_URL + 'IMD2020_Generative_Image_Inpainting_yu2018_05.zip',
    BASE_URL + 'IMD2020_Generative_Image_Inpainting_yu2018_06.zip',
    BASE_URL + 'IMD2020_Generative_Image_Inpainting_yu2018_07.zip',
]

# \u2500\u2500 Mask zip (for tampered images) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
MASK_ZIP = BASE_URL + 'IMD2020_Generative_Image_Inpainting_yu2018_mask.zip'

# \u2500\u2500 Download real images \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
print('=== Downloading Real Images ===')
for url in REAL_ZIPS:
    download_and_extract(url, REAL_DIR)

# \u2500\u2500 Download tampered images \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
print('\n=== Downloading Tampered (Inpainting) Images ===')
for url in FAKE_ZIPS:
    download_and_extract(url, FAKE_DIR)

# \u2500\u2500 Download masks \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
print('\n=== Downloading Masks ===')
download_and_extract(MASK_ZIP, MASK_DIR)

print('\n\u2705 All downloads complete!')

=== Downloading Real Images ===


    IMD2020_real_01.zip:   0%|          | 0.00/1.96G [00:00<?, ?B/s]

## 4. Scan Dataset & Build Records

In [ ]:
VALID_EXT = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
IMG_SIZE  = 256

def gather_images(folder):
    """Recursively collect all image files under folder."""
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if os.path.splitext(f)[1].lower() in VALID_EXT:
                paths.append(os.path.join(root, f))
    return sorted(paths)

real_imgs = gather_images(REAL_DIR)
fake_imgs = gather_images(FAKE_DIR)
mask_imgs = gather_images(MASK_DIR)

print(f'Real images   : {len(real_imgs):,}')
print(f'Tampered imgs : {len(fake_imgs):,}')
print(f'Masks found   : {len(mask_imgs):,}')

# ── Build a stem→mask lookup for pairing ──────────────────────────────────
# IMD2020 masks share the same filename stem as the tampered image.
mask_lookup = {}
for mp in mask_imgs:
    stem = Path(mp).stem
    mask_lookup[stem] = mp

print(f'\nMask lookup built: {len(mask_lookup):,} entries')
# Show a few samples
for k, v in list(mask_lookup.items())[:3]:
    print(f'  {k} → {v}')

In [ ]:
# ── Build records list ─────────────────────────────────────────────────────
all_records = []

# Real images → label 0, no mask
for img_path in real_imgs:
    all_records.append({'img': img_path, 'mask': None, 'label': 0})

# Tampered images → label 1, attempt mask match
matched_masks = 0
for img_path in fake_imgs:
    stem = Path(img_path).stem
    # Try exact match first, then partial
    mask_path = mask_lookup.get(stem, None)
    if mask_path is None:
        # Partial match: mask stem is substring of image stem or vice-versa
        for mk, mv in mask_lookup.items():
            if mk in stem or stem in mk:
                mask_path = mv
                break
    if mask_path:
        matched_masks += 1
    all_records.append({'img': img_path, 'mask': mask_path, 'label': 1})

print(f'Total records  : {len(all_records):,}')
print(f'Real           : {sum(1 for r in all_records if r["label"]==0):,}')
print(f'Tampered       : {sum(1 for r in all_records if r["label"]==1):,}')
print(f'Masks matched  : {matched_masks:,} / {len(fake_imgs):,}')

## 5. ELA Helper + Dataset + DataLoaders

In [ ]:
# ── ELA (Error Level Analysis) ─────────────────────────────────────────────
def convert_to_ela(path, quality=90):
    """Return ELA image as numpy array (H, W, 3) uint8."""
    try:
        image = Image.open(path).convert('RGB')
        buf   = io.BytesIO()
        image.save(buf, 'JPEG', quality=quality)
        buf.seek(0)
        temp     = Image.open(buf).copy()
        ela      = ImageChops.difference(image, temp)
        extrema  = ela.getextrema()
        max_diff = max(ex[1] for ex in extrema) or 1
        ela      = ImageEnhance.Brightness(ela).enhance(255.0 / max_diff)
        return np.array(ela)
    except Exception:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

def load_mask(mask_path, size):
    """Load binary mask; return zeros if none."""
    if mask_path and os.path.exists(mask_path):
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is not None:
            return cv2.resize(m, (size, size)).astype(np.float32) / 255.0
    return np.zeros((size, size), dtype=np.float32)

print('ELA helper ready.')

In [ ]:
# ── Albumentations augmentations ───────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
], additional_targets={'ela': 'image', 'mask': 'mask'})

val_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
], additional_targets={'ela': 'image', 'mask': 'mask'})

print('Augmentations ready.')

In [ ]:
# ── Dataset class ──────────────────────────────────────────────────────────
class ForgeryDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records   = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]

        # Load RGB image
        img = cv2.imread(rec['img'])
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        # ELA
        ela = convert_to_ela(rec['img'])
        ela = cv2.resize(ela, (IMG_SIZE, IMG_SIZE))

        # Mask
        mask = load_mask(rec['mask'], IMG_SIZE)

        if self.transform:
            aug  = self.transform(image=img, ela=ela, mask=mask)
            img  = aug['image']   # (3, H, W)
            ela  = aug['ela']     # (3, H, W)
            mask = aug['mask']    # (H, W)

        # Concatenate RGB + ELA → 6-channel input
        inp = torch.cat([img, ela], dim=0)   # (6, H, W)

        label = torch.tensor(rec['label'], dtype=torch.long)
        return inp, mask, label

print('Dataset class defined.')

In [ ]:
# ── Train / Val / Test split  (80 / 10 / 10) ──────────────────────────────
BATCH_SIZE = 16

indices = list(range(len(all_records)))
labels  = [r['label'] for r in all_records]

idx_train, idx_tmp = train_test_split(
    indices, test_size=0.20, random_state=SEED, stratify=labels)

labels_tmp = [labels[i] for i in idx_tmp]
idx_val, idx_test = train_test_split(
    idx_tmp, test_size=0.50, random_state=SEED, stratify=labels_tmp)

print(f'Train: {len(idx_train):,}  |  Val: {len(idx_val):,}  |  Test: {len(idx_test):,}')

train_ds = ForgeryDataset([all_records[i] for i in idx_train], transform=train_aug)
val_ds   = ForgeryDataset([all_records[i] for i in idx_val],   transform=val_aug)
test_ds  = ForgeryDataset([all_records[i] for i in idx_test],  transform=val_aug)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print('DataLoaders ready.')
print(f'  Train batches : {len(train_loader)}')
print(f'  Val   batches : {len(val_loader)}')
print(f'  Test  batches : {len(test_loader)}')

## 6. Model — ForgeryClassifier (ResNet34 Encoder)

In [ ]:
class ForgeryClassifier(nn.Module):
    """
    Classification model using a ResNet34 encoder from SMP.
    Input : (B, 6, H, W)  — RGB + ELA concatenated
    Output: cls_logits (B, 2)
    """
    def __init__(self, encoder_name='resnet34', pretrained=True):
        super().__init__()
        weights = 'imagenet' if pretrained else None
        # Use SMP Unet's encoder as a feature extractor
        temp_unet = smp.Unet(
            encoder_name    = encoder_name,
            encoder_weights = weights,
            in_channels     = 6,
            classes         = 1,
            activation      = None,
        )
        self.encoder = temp_unet.encoder

        enc_out_ch = self.encoder.out_channels[-1]  # 512 for resnet34
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(enc_out_ch, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        features   = self.encoder(x)
        bottleneck = features[-1]
        return self.cls_head(bottleneck)


model = ForgeryClassifier(encoder_name='resnet34', pretrained=True).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total:,}')

## 7. Load Previous Checkpoint

Upload `best_forgery_model (1).pth` via the Colab **Files panel** (📁 left sidebar) before running this cell.  
The cell auto-searches `/content/` and Google Drive — no path editing needed.

In [ ]:
import glob

# ── Auto-locate the checkpoint ─────────────────────────────────────────────
# Edit MANUAL_PATH below if your file is somewhere specific, otherwise
# leave as None and the auto-search will find it.
MANUAL_PATH = None  # e.g. '/content/drive/MyDrive/best_forgery_model (1).pth'

CKPT_PATH = MANUAL_PATH
if CKPT_PATH is None:
    candidates = [
        '/content/best_forgery_model (1).pth',
        '/content/best_forgery_model.pth',
        '/content/drive/MyDrive/best_forgery_model (1).pth',
        '/content/drive/MyDrive/best_forgery_model.pth',
    ] + glob.glob('/content/*.pth') + glob.glob('/content/drive/MyDrive/*.pth')

    for p in candidates:
        if os.path.exists(p):
            CKPT_PATH = p
            break

if CKPT_PATH is None:
    raise FileNotFoundError(
        '\n❌ Checkpoint not found!\n'
        'Please upload best_forgery_model (1).pth using the Files panel (📁)\n'
        'on the left sidebar, then re-run this cell.'
    )

print(f'✅ Found checkpoint: {CKPT_PATH}')

# ── Load weights — handles both raw state_dict and full checkpoint ─────────
raw = torch.load(CKPT_PATH, map_location=device)

if isinstance(raw, dict) and 'model_state_dict' in raw:
    state_dict = raw['model_state_dict']
    prev = raw.get('test_metrics', {})
    print(f'Full checkpoint. Previous metrics → {prev}')
else:
    # Saved as: torch.save(model.state_dict(), path)
    state_dict = raw
    print('Raw state_dict checkpoint.')

missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f'  ⚠️  Missing keys ({len(missing)}): {missing[:3]}')
if unexpected:
    print(f'  ⚠️  Unexpected keys ({len(unexpected)}): {unexpected[:3]}')

print('\n✅ Weights restored — ready to continue training on IMD2020.')

## 8. Loss, Optimizer & Scheduler

In [ ]:
cls_criterion = nn.CrossEntropyLoss()

# Lower LR than original training (5e-5 instead of 2e-4) because we are
# fine-tuning from an already-trained checkpoint, not training from scratch.
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5, eta_min=1e-7)

print('Loss, optimizer, scheduler ready.')
print('Fine-tuning LR: 5e-5 (4x lower than initial training)')

## 9. Training Loop (5 Epochs — Fine-tuning from Checkpoint)

In [ ]:
EPOCHS    = 5
BEST_VAL  = 0.0
BEST_PATH = 'best_forgery_model_IMD2020.pth'

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc' : [], 'val_acc' : []
}

for epoch in range(1, EPOCHS + 1):

    # ── TRAIN ──────────────────────────────────────────────────────────────
    model.train()
    t_loss = t_correct = t_total = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]', leave=False)
    for imgs, masks, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        cls_out  = model(imgs)
        loss     = cls_criterion(cls_out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        t_loss    += loss.item()
        _, pred    = cls_out.max(1)
        t_correct += pred.eq(labels).sum().item()
        t_total   += labels.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    train_loss = t_loss / len(train_loader)
    train_acc  = t_correct / t_total

    # ── VALIDATION ─────────────────────────────────────────────────────────
    model.eval()
    v_loss = v_correct = v_total = 0

    with torch.no_grad():
        for imgs, masks, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            cls_out  = model(imgs)
            loss     = cls_criterion(cls_out, labels)

            v_loss    += loss.item()
            _, pred    = cls_out.max(1)
            v_correct += pred.eq(labels).sum().item()
            v_total   += labels.size(0)

    val_loss = v_loss / len(val_loader)
    val_acc  = v_correct / v_total

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    saved_mark = ''
    if val_acc > BEST_VAL:
        BEST_VAL = val_acc
        torch.save(model.state_dict(), BEST_PATH)
        saved_mark = '  ✅ saved'

    print(f'Ep {epoch:02d}/{EPOCHS}  '
          f'Loss {train_loss:.4f}/{val_loss:.4f}  '
          f'Acc {train_acc:.4f}/{val_acc:.4f}'
          + saved_mark)

print(f'\nBest model saved to: {BEST_PATH}  (val_acc={BEST_VAL:.4f})')

## 10. Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], label='Train', marker='o')
axes[0].plot(epochs_range, history['val_loss'],   label='Val',   marker='o')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_range, history['train_acc'], label='Train', marker='o')
axes[1].plot(epochs_range, history['val_acc'],   label='Val',   marker='o')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True)

plt.suptitle('IMD2020 Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves_IMD2020.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves_IMD2020.png')

## 11. Test Set Evaluation & Confusion Matrix

In [ ]:
# Load best weights
model.load_state_dict(torch.load(BEST_PATH, map_location=device))
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for imgs, masks, labels in tqdm(test_loader, desc='Testing'):
        imgs = imgs.to(device)
        out  = model(imgs)
        _, preds = out.max(1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc   = accuracy_score(y_true, y_pred)
prec  = precision_score(y_true, y_pred, zero_division=0)
rec   = recall_score(y_true, y_pred, zero_division=0)
f1    = f1_score(y_true, y_pred, zero_division=0)

print('=' * 45)
print(f'  Test Accuracy  : {acc:.4f}')
print(f'  Precision      : {prec:.4f}')
print(f'  Recall         : {rec:.4f}')
print(f'  F1 Score       : {f1:.4f}')
print('=' * 45)
print()
print(classification_report(y_true, y_pred,
      target_names=['Authentic', 'Tampered']))

# ── Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Authentic', 'Tampered'],
            yticklabels=['Authentic', 'Tampered'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — IMD2020 Test Set')
plt.tight_layout()
plt.savefig('confusion_matrix_IMD2020.png', dpi=150)
plt.show()
print('Saved: confusion_matrix_IMD2020.png')

## 12. Sample Predictions with ELA Visualization

In [ ]:
def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor[:3] * std + mean).clamp(0, 1)

# Sample 8 images from test — mix of real and tampered
sample_real    = [all_records[i] for i in idx_test if all_records[i]['label'] == 0]
sample_tamp    = [all_records[i] for i in idx_test if all_records[i]['label'] == 1]
samples = random.sample(sample_tamp, min(4, len(sample_tamp))) + \
          random.sample(sample_real, min(4, len(sample_real)))
random.shuffle(samples)

model.eval()
n = len(samples)
fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
if n == 1:
    axes = [axes]

col_titles = ['Original Image', 'ELA', 'Prediction']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=13, fontweight='bold')

for row_idx, rec in enumerate(samples):
    img_rgb  = cv2.imread(rec['img'])
    if img_rgb is None:
        continue
    img_rgb  = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
    img_disp = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    ela_disp = convert_to_ela(rec['img'])
    ela_disp = cv2.resize(ela_disp, (IMG_SIZE, IMG_SIZE))
    mask_gt  = load_mask(rec['mask'], IMG_SIZE)

    aug = val_aug(image=img_disp, ela=ela_disp, mask=mask_gt)
    inp = torch.cat([aug['image'], aug['ela']], dim=0).unsqueeze(0).to(device)

    with torch.no_grad():
        cls_out = model(inp)
    _, cls_pred = cls_out.max(1)
    pred_label  = 'Tampered' if cls_pred.item() == 1 else 'Authentic'
    true_label  = 'Tampered' if rec['label'] == 1 else 'Authentic'
    correct     = '✅' if pred_label == true_label else '❌'

    ax_row = axes[row_idx]
    ax_row[0].imshow(img_disp)
    ax_row[0].set_ylabel(f'GT: {true_label}', fontsize=10)
    ax_row[0].axis('off')
    ax_row[1].imshow(ela_disp)
    ax_row[1].axis('off')
    ax_row[2].imshow(img_disp)
    ax_row[2].set_title(f'{correct} Pred: {pred_label}', fontsize=11)
    ax_row[2].axis('off')

plt.suptitle('IMD2020 — Sample Predictions', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('sample_predictions_IMD2020.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sample_predictions_IMD2020.png')

## 13. Save Full Checkpoint

In [ ]:
checkpoint = {
    'model_state_dict' : model.state_dict(),
    'optimizer_state'  : optimizer.state_dict(),
    'history'          : history,
    'test_metrics': {
        'accuracy'  : float(acc),
        'precision' : float(prec),
        'recall'    : float(rec),
        'f1'        : float(f1),
    },
    'config': {
        'encoder'    : 'resnet34',
        'img_size'   : IMG_SIZE,
        'epochs'     : EPOCHS,
        'batch_size' : BATCH_SIZE,
        'dataset'    : 'IMD2020 (Real + Inpainting)',
    }
}

FINAL_PATH = 'best_forgery_model_IMD2020_full.pth'
torch.save(checkpoint, FINAL_PATH)
print(f'Full checkpoint saved → {FINAL_PATH}')

# Also copy to Google Drive if mounted
try:
    drive_path = '/content/drive/MyDrive/'
    if os.path.isdir(drive_path):
        shutil.copy(FINAL_PATH, drive_path + FINAL_PATH)
        shutil.copy('confusion_matrix_IMD2020.png',
                    drive_path + 'confusion_matrix_IMD2020.png')
        shutil.copy('training_curves_IMD2020.png',
                    drive_path + 'training_curves_IMD2020.png')
        print('Files also copied to Google Drive.')
except Exception as e:
    print(f'(Drive copy skipped: {e})')

print('\n=== Training Complete ===')
print(f'  Best Val Acc : {BEST_VAL:.4f}')
print(f'  Test Acc     : {acc:.4f}')
print(f'  Test F1      : {f1:.4f}')